# ModernBERT Spoiler Detection

This notebook fine-tunes ModernBERT for spoiler detection on IMDb movie reviews and Goodreads book reviews.

We evaluate the model in four settings:

1. IMDb → IMDb (in-domain)
2. IMDb → Goodreads (cross-domain)
3. Goodreads → Goodreads (in-domain)
4. Goodreads → IMDb (cross-domain)

The same train, validation, and test splits as in the TF-IDF baseline are used to ensure a direct comparison between both approaches.


## Setup


In [1]:
!git clone -b dev https://github.com/rafaelTamm/NLE-Project-Spoiler-Detection.git
%cd NLE-Project-Spoiler-Detection


Cloning into 'NLE-Project-Spoiler-Detection'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 74 (delta 30), reused 20 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 31.49 MiB | 19.17 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/NLE-Project-Spoiler-Detection


In [2]:
!pip install -q datasets transformers accelerate


In [3]:
import os
import pandas as pd
import numpy as np
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


## Load Data


In [4]:
imdb_train = pd.read_csv("data/imdb_train.csv")
imdb_val = pd.read_csv("data/imdb_val.csv")
imdb_test = pd.read_csv("data/imdb_test.csv")

goodreads_train = pd.read_csv("data/goodreads_train.csv")
goodreads_val = pd.read_csv("data/goodreads_val.csv")
goodreads_test = pd.read_csv("data/goodreads_test.csv")

datasets = {
    "IMDb train": imdb_train,
    "IMDb val": imdb_val,
    "IMDb test": imdb_test,
    "Goodreads train": goodreads_train,
    "Goodreads val": goodreads_val,
    "Goodreads test": goodreads_test,
}

for name, df in datasets.items():
    print(f"{name}: {len(df)} reviews")


IMDb train: 13913 reviews
IMDb val: 2821 reviews
IMDb test: 3266 reviews
Goodreads train: 13982 reviews
Goodreads val: 2934 reviews
Goodreads test: 3084 reviews


## Prepare Hugging Face Datasets


In [5]:
imdb_dataset = DatasetDict({
    "train": Dataset.from_pandas(imdb_train),
    "validation": Dataset.from_pandas(imdb_val),
    "test": Dataset.from_pandas(imdb_test)
})

goodreads_dataset = DatasetDict({
    "train": Dataset.from_pandas(goodreads_train),
    "validation": Dataset.from_pandas(goodreads_val),
    "test": Dataset.from_pandas(goodreads_test)
})


## Prepare Labels


In [6]:
def prepare_labels(example):
    example["label"] = int(example["is_spoiler"])
    return example

imdb_dataset = imdb_dataset.map(prepare_labels)
goodreads_dataset = goodreads_dataset.map(prepare_labels)

print("IMDb:", imdb_dataset["train"][0]["is_spoiler"], "->", imdb_dataset["train"][0]["label"])
print("Goodreads:", goodreads_dataset["train"][0]["is_spoiler"], "->", goodreads_dataset["train"][0]["label"])


Map:   0%|          | 0/13913 [00:00<?, ? examples/s]

Map:   0%|          | 0/2821 [00:00<?, ? examples/s]

Map:   0%|          | 0/3266 [00:00<?, ? examples/s]

Map:   0%|          | 0/13982 [00:00<?, ? examples/s]

Map:   0%|          | 0/2934 [00:00<?, ? examples/s]

Map:   0%|          | 0/3084 [00:00<?, ? examples/s]

IMDb: False -> 0
Goodreads: False -> 0


## ModernBERT Tokenizer

The previous length analysis showed that a maximum sequence length of 1024 preserves substantially more spoiler-review content than 512 while keeping training feasible.


In [7]:
checkpoint = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

MAX_LENGTH = 1024


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

## Tokenization


In [8]:
def tokenize_function(examples):
    return tokenizer(
        examples["review_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_imdb = imdb_dataset.map(tokenize_function, batched=True)
tokenized_goodreads = goodreads_dataset.map(tokenize_function, batched=True)

print(tokenized_imdb["train"][0].keys())
print("IMDb token count:", len(tokenized_imdb["train"][0]["input_ids"]))
print("Goodreads token count:", len(tokenized_goodreads["train"][0]["input_ids"]))


Map:   0%|          | 0/13913 [00:00<?, ? examples/s]

Map:   0%|          | 0/2821 [00:00<?, ? examples/s]

Map:   0%|          | 0/3266 [00:00<?, ? examples/s]

Map:   0%|          | 0/13982 [00:00<?, ? examples/s]

Map:   0%|          | 0/2934 [00:00<?, ? examples/s]

Map:   0%|          | 0/3084 [00:00<?, ? examples/s]

dict_keys(['review_text', 'is_spoiler', 'item_id', 'label', 'input_ids', 'attention_mask'])
IMDb token count: 138
Goodreads token count: 71


## Dynamic Padding


In [9]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## Evaluation Metrics


In [10]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0)
    }


## GPU Check


In [11]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU available.")


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Google Drive Checkpoints


In [12]:
from google.colab import drive
drive.mount("/content/drive")

IMDB_OUTPUT_DIR = "/content/drive/MyDrive/NLP_Spoiler_Project/modernbert_imdb"
os.makedirs(IMDB_OUTPUT_DIR, exist_ok=True)

print("Checkpoint directory:", IMDB_OUTPUT_DIR)
print("Exists:", os.path.exists(IMDB_OUTPUT_DIR))


Mounted at /content/drive
Checkpoint directory: /content/drive/MyDrive/NLP_Spoiler_Project/modernbert_imdb
Exists: True


## IMDb Model


In [13]:
imdb_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## IMDb Training Setup


In [14]:
imdb_training_args = TrainingArguments(
    output_dir=IMDB_OUTPUT_DIR,
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    report_to="none"
)

imdb_trainer = Trainer(
    model=imdb_model,
    args=imdb_training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


## IMDb Fine-Tuning

**Stop here until the runtime/GPU is ready for the final training run.**

The test sets are not used for model selection.


In [15]:
# Sanity check: no movie/book appears in more than one split

def check_split_overlap(train_df, val_df, test_df, name):
    train_ids = set(train_df["item_id"])
    val_ids = set(val_df["item_id"])
    test_ids = set(test_df["item_id"])

    print(name)
    print("Train / Validation overlap:", len(train_ids & val_ids))
    print("Train / Test overlap:", len(train_ids & test_ids))
    print("Validation / Test overlap:", len(val_ids & test_ids))


check_split_overlap(imdb_train, imdb_val, imdb_test, "IMDb")
check_split_overlap(
    goodreads_train,
    goodreads_val,
    goodreads_test,
    "Goodreads"
)

IMDb
Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0
Goodreads
Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0


In [ ]:
imdb_train_result = imdb_trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1
1,2.000170,0.559371,0.713577,0.781060,0.617869,0.689946,0.711903
2,1.944008,0.619562,0.714286,0.746393,0.675601,0.709235,0.714199
3,1.113793,1.474191,0.703651,0.720599,0.694845,0.707488,0.703600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## IMDb Evaluation


In [16]:
imdb_in_domain = imdb_trainer.evaluate(tokenized_imdb["test"])
imdb_to_goodreads = imdb_trainer.evaluate(tokenized_goodreads["test"])

print("IMDb → IMDb:", imdb_in_domain)
print("IMDb → Goodreads:", imdb_to_goodreads)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
No log,0.721675,0,0.497244,0.427027,0.048916,0.087778,0.370389


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
No log,0.735080,0,0.415370,0.295209,0.124187,0.174828,0.361077


IMDb → IMDb: {'eval_loss': 0.7216746211051941, 'eval_accuracy': 0.4972443355786895, 'eval_precision': 0.42702702702702705, 'eval_recall': 0.04891640866873065, 'eval_f1': 0.08777777777777777, 'eval_macro_f1': 0.3703893115431577}
IMDb → Goodreads: {'eval_loss': 0.7350800037384033, 'eval_accuracy': 0.4153696498054475, 'eval_precision': 0.29520865533230295, 'eval_recall': 0.12418725617685306, 'eval_f1': 0.1748283752860412, 'eval_macro_f1': 0.36107725568218707}


## Goodreads Fine-Tuning

A fresh ModernBERT model is fine-tuned on the Goodreads training split.
The Goodreads validation split is used for model selection.

The selected model is evaluated on:
- Goodreads test (in-domain)
- IMDb test (cross-domain)

In [17]:
GOODREADS_OUTPUT_DIR = "/content/drive/MyDrive/NLP_Spoiler_Project/modernbert_goodreads"

os.makedirs(GOODREADS_OUTPUT_DIR, exist_ok=True)

print("Goodreads checkpoint directory:", GOODREADS_OUTPUT_DIR)
print("Exists:", os.path.exists(GOODREADS_OUTPUT_DIR))

Goodreads checkpoint directory: /content/drive/MyDrive/NLP_Spoiler_Project/modernbert_goodreads
Exists: True


In [18]:
goodreads_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
goodreads_training_args = TrainingArguments(
    output_dir=GOODREADS_OUTPUT_DIR,
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    report_to="none"
)

In [20]:
goodreads_trainer = Trainer(
    model=goodreads_model,
    args=goodreads_training_args,
    train_dataset=tokenized_goodreads["train"],
    eval_dataset=tokenized_goodreads["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [20]:
goodreads_train_result = goodreads_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1
1,1.555735,0.367688,0.847307,0.815691,0.896033,0.853977,0.846988
2,1.139061,0.474401,0.855147,0.844061,0.870041,0.856854,0.855126
3,0.442940,1.010789,0.851738,0.838944,0.869357,0.853880,0.851706


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Goodreads Evaluation

The selected Goodreads model is evaluated on:
- Goodreads test (in-domain)
- IMDb test (cross-domain)

In [21]:
goodreads_in_domain = goodreads_trainer.evaluate(
    tokenized_goodreads["test"]
)

goodreads_to_imdb = goodreads_trainer.evaluate(
    tokenized_imdb["test"]
)

print("Goodreads → Goodreads:")
print(goodreads_in_domain)

print("\nGoodreads → IMDb:")
print(goodreads_to_imdb)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
No log,0.786340,0,0.516537,0.507779,0.997399,0.672955,0.373146


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Macro F1
No log,0.895423,0,0.494795,0.494640,1.000000,0.661885,0.331548


Goodreads → Goodreads:
{'eval_loss': 0.7863403558731079, 'eval_accuracy': 0.5165369649805448, 'eval_precision': 0.5077788811651771, 'eval_recall': 0.9973992197659298, 'eval_f1': 0.6729545953059881, 'eval_macro_f1': 0.3731460359997933}

Goodreads → IMDb:
{'eval_loss': 0.8954229950904846, 'eval_accuracy': 0.4947948560930802, 'eval_precision': 0.4946401225114854, 'eval_recall': 1.0, 'eval_f1': 0.6618852459016393, 'eval_macro_f1': 0.331547949827333}


## ModernBERT Results

The following table summarizes the in-domain and cross-domain evaluation results for both fine-tuned ModernBERT models.

In [22]:
modernbert_results = pd.DataFrame([
    {
        "Experiment": "IMDb → IMDb",
        "Accuracy": 0.7317819963257808,
        "Precision": 0.7432521395655036,
        "Recall": 0.6990712074303406,
        "F1": 0.7204850031908104,
        "Macro F1": 0.7313431490350757,
    },
    {
        "Experiment": "IMDb → Goodreads",
        "Accuracy": 0.7311932555123216,
        "Precision": 0.6948873007146784,
        "Recall": 0.8218465539661899,
        "F1": 0.7530533214179327,
        "Macro F1": 0.7290702395065473,
    },
    {
        "Experiment": "Goodreads → Goodreads",
        "Accuracy": goodreads_in_domain["eval_accuracy"],
        "Precision": goodreads_in_domain["eval_precision"],
        "Recall": goodreads_in_domain["eval_recall"],
        "F1": goodreads_in_domain["eval_f1"],
        "Macro F1": goodreads_in_domain["eval_macro_f1"],
    },
    {
        "Experiment": "Goodreads → IMDb",
        "Accuracy": goodreads_to_imdb["eval_accuracy"],
        "Precision": goodreads_to_imdb["eval_precision"],
        "Recall": goodreads_to_imdb["eval_recall"],
        "F1": goodreads_to_imdb["eval_f1"],
        "Macro F1": goodreads_to_imdb["eval_macro_f1"],
    },
])

modernbert_results.round(4)

,Experiment,Accuracy,Precision,Recall,F1,Macro F1
0,IMDb → IMDb,0.7318,0.7433,0.6991,0.7205,0.7313
1,IMDb → Goodreads,0.7312,0.6949,0.8218,0.7531,0.7291
2,Goodreads → Goodreads,0.5165,0.5078,0.9974,0.6730,0.3731
3,Goodreads → IMDb,0.4948,0.4946,1.0000,0.6619,0.3315


In [23]:
imdb_to_goodreads_drop = (
    modernbert_results.loc[0, "Macro F1"]
    - modernbert_results.loc[1, "Macro F1"]
)

goodreads_to_imdb_drop = (
    modernbert_results.loc[2, "Macro F1"]
    - modernbert_results.loc[3, "Macro F1"]
)

print(f"IMDb → Goodreads Macro F1 drop: {imdb_to_goodreads_drop:.4f}")
print(f"Goodreads → IMDb Macro F1 drop: {goodreads_to_imdb_drop:.4f}")

IMDb → Goodreads Macro F1 drop: 0.0023
Goodreads → IMDb Macro F1 drop: 0.0416


## ModernBERT Cross-Domain Error Analysis

We analyze false positives and false negatives for both cross-domain directions:
IMDb → Goodreads and Goodreads → IMDb.

In [24]:
import os

IMDB_OUTPUT_DIR = "/content/drive/MyDrive/NLP_Spoiler_Project/modernbert_imdb"
GOODREADS_OUTPUT_DIR = "/content/drive/MyDrive/NLP_Spoiler_Project/modernbert_goodreads"

print("IMDb checkpoints:")
print(os.listdir(IMDB_OUTPUT_DIR))

print("\nGoodreads checkpoints:")
print(os.listdir(GOODREADS_OUTPUT_DIR))

IMDb checkpoints:
['checkpoint-3480', 'checkpoint-5220']

Goodreads checkpoints:
['checkpoint-3496', 'checkpoint-5244']


In [25]:
from transformers import AutoModelForSequenceClassification

IMDB_BEST_CHECKPOINT = os.path.join(
    IMDB_OUTPUT_DIR,
    "checkpoint-3480"
)

GOODREADS_BEST_CHECKPOINT = os.path.join(
    GOODREADS_OUTPUT_DIR,
    "checkpoint-3496"
)

imdb_model = AutoModelForSequenceClassification.from_pretrained(
    IMDB_BEST_CHECKPOINT
)

goodreads_model = AutoModelForSequenceClassification.from_pretrained(
    GOODREADS_BEST_CHECKPOINT
)

print("IMDb model loaded from:", IMDB_BEST_CHECKPOINT)
print("Goodreads model loaded from:", GOODREADS_BEST_CHECKPOINT)

Loading weights:   0%|          | 0/138 [00:01<?, ?it/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

IMDb model loaded from: /content/drive/MyDrive/NLP_Spoiler_Project/modernbert_imdb/checkpoint-3480
Goodreads model loaded from: /content/drive/MyDrive/NLP_Spoiler_Project/modernbert_goodreads/checkpoint-3496


In [26]:
from transformers import Trainer

# IMDb-trained model -> Goodreads test set
imdb_error_trainer = Trainer(
    model=imdb_model,
    data_collator=data_collator,
    processing_class=tokenizer,
)

imdb_to_goodreads_predictions = imdb_error_trainer.predict(
    tokenized_goodreads["test"]
)

# Goodreads-trained model -> IMDb test set
goodreads_error_trainer = Trainer(
    model=goodreads_model,
    data_collator=data_collator,
    processing_class=tokenizer,
)

goodreads_to_imdb_predictions = goodreads_error_trainer.predict(
    tokenized_imdb["test"]
)

print("Cross-domain predictions completed.")

Cross-domain predictions completed.


In [27]:
import numpy as np

# Convert logits to predicted class labels
imdb_to_goodreads_pred_labels = np.argmax(
    imdb_to_goodreads_predictions.predictions,
    axis=-1
)

goodreads_to_imdb_pred_labels = np.argmax(
    goodreads_to_imdb_predictions.predictions,
    axis=-1
)

# Create copies of the original test DataFrames
goodreads_errors = goodreads_test.copy()
imdb_errors = imdb_test.copy()

# Add model predictions
goodreads_errors["prediction"] = imdb_to_goodreads_pred_labels
imdb_errors["prediction"] = goodreads_to_imdb_pred_labels

# Convert true labels to 0/1
goodreads_errors["label"] = goodreads_errors["is_spoiler"].astype(int)
imdb_errors["label"] = imdb_errors["is_spoiler"].astype(int)

# Extract false positives and false negatives
imdb_to_goodreads_fp = goodreads_errors[
    (goodreads_errors["label"] == 0) &
    (goodreads_errors["prediction"] == 1)
]

imdb_to_goodreads_fn = goodreads_errors[
    (goodreads_errors["label"] == 1) &
    (goodreads_errors["prediction"] == 0)
]

goodreads_to_imdb_fp = imdb_errors[
    (imdb_errors["label"] == 0) &
    (imdb_errors["prediction"] == 1)
]

goodreads_to_imdb_fn = imdb_errors[
    (imdb_errors["label"] == 1) &
    (imdb_errors["prediction"] == 0)
]

print("IMDb → Goodreads")
print("False Positives:", len(imdb_to_goodreads_fp))
print("False Negatives:", len(imdb_to_goodreads_fn))

print("\nGoodreads → IMDb")
print("False Positives:", len(goodreads_to_imdb_fp))
print("False Negatives:", len(goodreads_to_imdb_fn))

IMDb → Goodreads
False Positives: 555
False Negatives: 274

Goodreads → IMDb
False Positives: 417
False Negatives: 682


In [28]:
ERROR_SAMPLE_SIZE = 10
RANDOM_STATE = 42

error_samples = pd.concat([
    imdb_to_goodreads_fp.sample(
        n=ERROR_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    ).assign(
        experiment="IMDb → Goodreads",
        error_type="False Positive"
    ),

    imdb_to_goodreads_fn.sample(
        n=ERROR_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    ).assign(
        experiment="IMDb → Goodreads",
        error_type="False Negative"
    ),

    goodreads_to_imdb_fp.sample(
        n=ERROR_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    ).assign(
        experiment="Goodreads → IMDb",
        error_type="False Positive"
    ),

    goodreads_to_imdb_fn.sample(
        n=ERROR_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    ).assign(
        experiment="Goodreads → IMDb",
        error_type="False Negative"
    )
], ignore_index=True)

error_samples[
    ["experiment", "error_type", "label", "prediction", "review_text"]
]

,experiment,error_type,label,prediction,review_text
0,IMDb → Goodreads,False Positive,0,1,The story follows Annalisa Townsend who is sai...
1,IMDb → Goodreads,False Positive,0,1,Anya Whitson is a cold woman. She has never lo...
2,IMDb → Goodreads,False Positive,0,1,3.75 stars I received an e-ARC copy of To Have...
3,IMDb → Goodreads,False Positive,0,1,"Ramblings. Yes, I'm still in my ""prolonging Th..."
4,IMDb → Goodreads,False Positive,0,1,"Okay, so I may have over hyped myself with thi..."
5,IMDb → Goodreads,False Positive,0,1,This is the story of Lila whom started her lif...
6,IMDb → Goodreads,False Positive,0,1,"After reading the second novella, I thought pe..."
7,IMDb → Goodreads,False Positive,0,1,Mmm I wasn't a super big fan of this one. The ...
8,IMDb → Goodreads,False Positive,0,1,I DNF at 26% because the plot was just too dra...
9,IMDb → Goodreads,False Positive,0,1,I stumbled upon Whitney's When You Were Here a...


In [29]:
error_samples.to_csv(
    "modernbert_cross_domain_error_sample.csv",
    index=False
)

print("Saved 40 error-analysis examples.")

Saved 40 error-analysis examples.


### Qualitative Error Analysis

To better understand the asymmetric cross-domain performance, we
randomly sampled 10 false positives and 10 false negatives from each
transfer direction (40 examples in total, random seed 42).

Several recurring patterns were observed. False positives frequently
contained detailed plot descriptions, character relationships, or
specific story events despite being labeled as non-spoilers. This
suggests that plot-related language alone is not a reliable indicator
of the spoiler label across domains.

False negatives often resembled ordinary evaluative reviews, with
spoiler information embedded within otherwise general discussion.
In the Goodreads → IMDb direction, even an example containing an
explicit spoiler warning ("HERE MIGHT BE SPOILERS") was classified as
a non-spoiler.

These examples suggest that the cross-domain errors cannot be explained
only by different vocabulary or writing styles. Differences in how
spoiler-relevant information is expressed and annotated across the two
datasets may also contribute to the observed transfer gap. Since this
analysis is based on a small qualitative sample, these patterns should
be interpreted as observations rather than general conclusions.

## Cross-Domain Confidence Analysis

We analyze the confidence of ModernBERT's cross-domain predictions.
This allows us to compare the model's confidence for correct and incorrect
predictions and to identify high-confidence errors.

In [30]:
import numpy as np
import pandas as pd
from scipy.special import softmax

# Convert logits to probabilities
imdb_to_goodreads_probs = softmax(
    imdb_to_goodreads_predictions.predictions,
    axis=1
)

goodreads_to_imdb_probs = softmax(
    goodreads_to_imdb_predictions.predictions,
    axis=1
)

# Confidence = probability assigned to the predicted class
imdb_to_goodreads_confidence = np.max(imdb_to_goodreads_probs, axis=1)
goodreads_to_imdb_confidence = np.max(goodreads_to_imdb_probs, axis=1)

print("IMDb → Goodreads")
print("Mean confidence:", imdb_to_goodreads_confidence.mean())

print("\nGoodreads → IMDb")
print("Mean confidence:", goodreads_to_imdb_confidence.mean())

IMDb → Goodreads
Mean confidence: 0.86256635

Goodreads → IMDb
Mean confidence: 0.9077348


In [31]:
# True labels
imdb_to_goodreads_labels = imdb_to_goodreads_predictions.label_ids
goodreads_to_imdb_labels = goodreads_to_imdb_predictions.label_ids

# Predicted labels
imdb_to_goodreads_preds = np.argmax(
    imdb_to_goodreads_predictions.predictions,
    axis=1
)

goodreads_to_imdb_preds = np.argmax(
    goodreads_to_imdb_predictions.predictions,
    axis=1
)

# Correct / incorrect masks
imdb_to_goodreads_correct = (
    imdb_to_goodreads_preds == imdb_to_goodreads_labels
)

goodreads_to_imdb_correct = (
    goodreads_to_imdb_preds == goodreads_to_imdb_labels
)

print("IMDb → Goodreads")
print(
    "Correct predictions:",
    imdb_to_goodreads_confidence[imdb_to_goodreads_correct].mean()
)
print(
    "Incorrect predictions:",
    imdb_to_goodreads_confidence[~imdb_to_goodreads_correct].mean()
)

print("\nGoodreads → IMDb")
print(
    "Correct predictions:",
    goodreads_to_imdb_confidence[goodreads_to_imdb_correct].mean()
)
print(
    "Incorrect predictions:",
    goodreads_to_imdb_confidence[~goodreads_to_imdb_correct].mean()
)

IMDb → Goodreads
Correct predictions: 0.8874425
Incorrect predictions: 0.7948997

Goodreads → IMDb
Correct predictions: 0.9230353
Incorrect predictions: 0.8775655


In [32]:
threshold = 0.90

imdb_to_goodreads_high_conf_errors = (
    (~imdb_to_goodreads_correct) &
    (imdb_to_goodreads_confidence >= threshold)
)

goodreads_to_imdb_high_conf_errors = (
    (~goodreads_to_imdb_correct) &
    (goodreads_to_imdb_confidence >= threshold)
)

print("IMDb → Goodreads")
print("Incorrect predictions:", (~imdb_to_goodreads_correct).sum())
print("High-confidence errors:", imdb_to_goodreads_high_conf_errors.sum())
print(
    "Share of errors with confidence >= 0.90:",
    imdb_to_goodreads_high_conf_errors.sum()
    / (~imdb_to_goodreads_correct).sum()
)

print("\nGoodreads → IMDb")
print("Incorrect predictions:", (~goodreads_to_imdb_correct).sum())
print("High-confidence errors:", goodreads_to_imdb_high_conf_errors.sum())
print(
    "Share of errors with confidence >= 0.90:",
    goodreads_to_imdb_high_conf_errors.sum()
    / (~goodreads_to_imdb_correct).sum()
)

IMDb → Goodreads
Incorrect predictions: 829
High-confidence errors: 271
Share of errors with confidence >= 0.90: 0.32689987937273823

Goodreads → IMDb
Incorrect predictions: 1099
High-confidence errors: 675
Share of errors with confidence >= 0.90: 0.6141947224749773


## Cross-Domain Performance by Review Length

We analyze whether cross-domain performance changes with review length.
Reviews are grouped by their tokenized length to examine whether short and
long reviews differ in classification performance.

In [33]:
from sklearn.metrics import accuracy_score, recall_score, f1_score

def performance_by_length(dataset, predictions, tokenizer):
    texts = dataset["review_text"]
    labels = np.array(dataset["label"])

    # Token length before truncation
    lengths = np.array([
        len(tokenizer(text, truncation=False)["input_ids"])
        for text in texts
    ])

    # Length groups based on the previous dataset analysis
    bins = [0, 128, 256, 512, 1024, np.inf]
    names = ["≤128", "129–256", "257–512", "513–1024", ">1024"]

    rows = []

    for lower, upper, name in zip(bins[:-1], bins[1:], names):
        mask = (lengths > lower) & (lengths <= upper)

        y_true = labels[mask]
        y_pred = predictions[mask]

        rows.append({
            "Length": name,
            "N": mask.sum(),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "Macro F1": f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
        })

    return pd.DataFrame(rows)


imdb_to_goodreads_length = performance_by_length(
    tokenized_goodreads["test"],
    imdb_to_goodreads_preds,
    tokenizer
)

goodreads_to_imdb_length = performance_by_length(
    tokenized_imdb["test"],
    goodreads_to_imdb_preds,
    tokenizer
)

print("IMDb → Goodreads")
display(imdb_to_goodreads_length)

print("\nGoodreads → IMDb")
display(goodreads_to_imdb_length)

IMDb → Goodreads


,Length,N,Accuracy,Recall,Macro F1
0,≤128,978,0.830266,0.429245,0.709877
1,129–256,681,0.696035,0.782609,0.683019
2,257–512,782,0.662404,0.897177,0.563664
3,513–1024,517,0.690522,0.956395,0.531830
4,>1024,126,0.746032,0.978947,0.456017



Goodreads → IMDb


,Length,N,Accuracy,Recall,Macro F1
0,≤128,322,0.760870,0.307692,0.574826
1,129–256,1246,0.659711,0.437008,0.625218
2,257–512,978,0.620654,0.583916,0.619173
3,513–1024,584,0.669521,0.718421,0.644476
4,>1024,136,0.750000,0.854369,0.644854


In [34]:
def label_distribution_by_length(dataset, tokenizer):
    texts = dataset["review_text"]
    labels = np.array(dataset["label"])

    lengths = np.array([
        len(tokenizer(text, truncation=False)["input_ids"])
        for text in texts
    ])

    bins = [0, 128, 256, 512, 1024, np.inf]
    names = ["≤128", "129–256", "257–512", "513–1024", ">1024"]

    rows = []

    for lower, upper, name in zip(bins[:-1], bins[1:], names):
        mask = (lengths > lower) & (lengths <= upper)
        bin_labels = labels[mask]

        rows.append({
            "Length": name,
            "N": len(bin_labels),
            "Spoilers": (bin_labels == 1).sum(),
            "Non-spoilers": (bin_labels == 0).sum(),
            "Spoiler share": (bin_labels == 1).mean()
        })

    return pd.DataFrame(rows)


print("Goodreads test set")
display(
    label_distribution_by_length(
        tokenized_goodreads["test"],
        tokenizer
    )
)

print("\nIMDb test set")
display(
    label_distribution_by_length(
        tokenized_imdb["test"],
        tokenizer
    )
)

Goodreads test set


,Length,N,Spoilers,Non-spoilers,Spoiler share
0,≤128,978,212,766,0.216769
1,129–256,681,391,290,0.574156
2,257–512,782,496,286,0.634271
3,513–1024,517,344,173,0.665377
4,>1024,126,95,31,0.753968



IMDb test set


,Length,N,Spoilers,Non-spoilers,Spoiler share
0,≤128,322,52,270,0.161491
1,129–256,1246,508,738,0.407705
2,257–512,978,572,406,0.584867
3,513–1024,584,380,204,0.650685
4,>1024,136,103,33,0.757353


## Save Analysis Results

The cross-domain confidence and review-length analyses are saved for use in the final results notebook.

In [35]:
import os

os.makedirs("results", exist_ok=True)

# Confidence summary
confidence_results = pd.DataFrame([
    {
        "Experiment": "IMDb → Goodreads",
        "Mean Confidence": imdb_to_goodreads_confidence.mean(),
        "Correct Confidence": imdb_to_goodreads_confidence[imdb_to_goodreads_correct].mean(),
        "Incorrect Confidence": imdb_to_goodreads_confidence[~imdb_to_goodreads_correct].mean(),
        "Errors": (~imdb_to_goodreads_correct).sum(),
        "High Confidence Errors": imdb_to_goodreads_high_conf_errors.sum(),
        "High Confidence Error Share": (
            imdb_to_goodreads_high_conf_errors.sum()
            / (~imdb_to_goodreads_correct).sum()
        )
    },
    {
        "Experiment": "Goodreads → IMDb",
        "Mean Confidence": goodreads_to_imdb_confidence.mean(),
        "Correct Confidence": goodreads_to_imdb_confidence[goodreads_to_imdb_correct].mean(),
        "Incorrect Confidence": goodreads_to_imdb_confidence[~goodreads_to_imdb_correct].mean(),
        "Errors": (~goodreads_to_imdb_correct).sum(),
        "High Confidence Errors": goodreads_to_imdb_high_conf_errors.sum(),
        "High Confidence Error Share": (
            goodreads_to_imdb_high_conf_errors.sum()
            / (~goodreads_to_imdb_correct).sum()
        )
    }
])

confidence_results.to_csv(
    "results/modernbert_confidence.csv",
    index=False
)

# Length results
imdb_to_goodreads_length_export = imdb_to_goodreads_length.copy()
imdb_to_goodreads_length_export["Experiment"] = "IMDb → Goodreads"

goodreads_to_imdb_length_export = goodreads_to_imdb_length.copy()
goodreads_to_imdb_length_export["Experiment"] = "Goodreads → IMDb"

length_results = pd.concat(
    [
        imdb_to_goodreads_length_export,
        goodreads_to_imdb_length_export
    ],
    ignore_index=True
)

length_results.to_csv(
    "results/modernbert_length_analysis.csv",
    index=False
)

print("Saved:")
print("results/modernbert_confidence.csv")
print("results/modernbert_length_analysis.csv")

Saved:
results/modernbert_confidence.csv
results/modernbert_length_analysis.csv


In [36]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def calculate_result(y_true, y_pred, experiment):
    return {
        "Experiment": experiment,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Macro F1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    }


# Cross-domain results from the predictions generated above
imdb_to_goodreads_result = calculate_result(
    imdb_to_goodreads_predictions.label_ids,
    imdb_to_goodreads_preds,
    "IMDb → Goodreads"
)

goodreads_to_imdb_result = calculate_result(
    goodreads_to_imdb_predictions.label_ids,
    goodreads_to_imdb_preds,
    "Goodreads → IMDb"
)

print(imdb_to_goodreads_result)
print(goodreads_to_imdb_result)

{'Experiment': 'IMDb → Goodreads', 'Accuracy': 0.7311932555123216, 'Precision': 0.6948873007146784, 'Recall': 0.8218465539661899, 'F1': 0.7530533214179327, 'Macro F1': 0.7290702395065473}
{'Experiment': 'Goodreads → IMDb', 'Accuracy': 0.6635027556644213, 'Precision': 0.6911111111111111, 'Recall': 0.5777089783281734, 'F1': 0.6293423271500843, 'Macro F1': 0.6606201403061888}


In [41]:
# In-domain predictions with the loaded best-checkpoint models

imdb_in_domain_predictions = imdb_error_trainer.predict(
    tokenized_imdb["test"]
)

goodreads_in_domain_predictions = goodreads_error_trainer.predict(
    tokenized_goodreads["test"]
)

imdb_in_domain_preds = np.argmax(
    imdb_in_domain_predictions.predictions,
    axis=1
)

goodreads_in_domain_preds = np.argmax(
    goodreads_in_domain_predictions.predictions,
    axis=1
)

imdb_in_domain_result = calculate_result(
    imdb_in_domain_predictions.label_ids,
    imdb_in_domain_preds,
    "IMDb → IMDb"
)

goodreads_in_domain_result = calculate_result(
    goodreads_in_domain_predictions.label_ids,
    goodreads_in_domain_preds,
    "Goodreads → Goodreads"
)

print(imdb_in_domain_result)
print(goodreads_in_domain_result)

{'Experiment': 'IMDb → IMDb', 'Accuracy': 0.7317819963257808, 'Precision': 0.7432521395655036, 'Recall': 0.6990712074303406, 'F1': 0.7204850031908104, 'Macro F1': 0.7313431490350757}
{'Experiment': 'Goodreads → Goodreads', 'Accuracy': 0.8527885862516212, 'Precision': 0.8510362694300518, 'Recall': 0.8543563068920677, 'F1': 0.8526930564568462, 'Macro F1': 0.8527885243398943}


In [42]:
# Combine and save all ModernBERT results

modernbert_results = pd.DataFrame([
    imdb_in_domain_result,
    imdb_to_goodreads_result,
    goodreads_in_domain_result,
    goodreads_to_imdb_result
])

display(modernbert_results)

modernbert_results.to_csv(
    "results/modernbert_results.csv",
    index=False
)

print("Saved: results/modernbert_results.csv")

,Experiment,Accuracy,Precision,Recall,F1,Macro F1
0,IMDb → IMDb,0.731782,0.743252,0.699071,0.720485,0.731343
1,IMDb → Goodreads,0.731193,0.694887,0.821847,0.753053,0.729070
2,Goodreads → Goodreads,0.852789,0.851036,0.854356,0.852693,0.852789
3,Goodreads → IMDb,0.663503,0.691111,0.577709,0.629342,0.660620


Saved: results/modernbert_results.csv
